In [ ]:
import requests
study_id = "study_id" # enter study id here
# sends a POST request to the Beiwe server to get participant table data
response = requests.post( 
    "https://studies.beiwe.org/get-summary-statistics/v1",
        data={
            "access_key": "BEIWE_ACCESS_KEY", # enter access key here
            "secret_key": "BEIWE_SECRET_KEY", # enter secret key here
            "study_id": study_id,
    },
    allow_redirects=False,
)
# if the request succeeds, save the returned CSV to disk
if response.status_code == 200:
    summary_statistics_table = f"summary_statistics_table_{study_id}.json"
    print(f"The data has been written to {summary_statistics_table}.")
    with open(summary_statistics_table, "wb") as f:
        f.write(response.content)
# otherwise, print the error returned by the server
else:
    print(response.content)
    print(f"There was an error downloading the data; The request returned an HTTP error of {response.status_code}")

The data has been written to summary_statistics_table_XXb2Gm78TVBGesqyd6oPjjXQ.json.


In [5]:
import pandas as pd
with open(summary_statistics_table, "rb") as f:
    df = pd.read_json(f)
    # this will not work unless the summary_statistics_table variable in cell above has been populated and the file exists

/var/folders/_0/zkvq7tgx2rb60mq9dkgh5hf00000gn/T/ipykernel_44474/3030540094.py:3: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df = pd.read_json(f)


#### This first part imports the pandas library and reads in the CSV file.

In [6]:
df.head()

,date,study_id,timezone,beiwe_accelerometer_bytes,beiwe_ambient_audio_bytes,beiwe_app_log_bytes,beiwe_bluetooth_bytes,beiwe_calls_bytes,beiwe_devicemotion_bytes,beiwe_gps_bytes,...,sycamore_total_surveys,sycamore_total_completed_surveys,sycamore_total_opened_surveys,sycamore_average_time_to_submit,sycamore_average_time_to_open,sycamore_average_duration,oak_walking_time,oak_steps,oak_cadence,participant_id
0,2026-01-24,XXb2Gm78TVBGesqyd6oPjjXQ,EST,27308369.0,NaN,NaN,NaN,NaN,15778588.0,72786.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,udr3p8nm
1,2026-01-24,XXb2Gm78TVBGesqyd6oPjjXQ,EST,18608486.0,NaN,NaN,NaN,NaN,10428031.0,58560.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,n32g5d2i
2,2026-01-24,XXb2Gm78TVBGesqyd6oPjjXQ,EST,13621074.0,NaN,NaN,NaN,NaN,7449652.0,43349.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,uo7y21p4
3,2026-01-24,XXb2Gm78TVBGesqyd6oPjjXQ,EST,143390631.0,NaN,695549.0,NaN,NaN,NaN,455163.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,3p4eblir
4,2026-01-24,XXb2Gm78TVBGesqyd6oPjjXQ,EST,26250022.0,NaN,388252.0,NaN,NaN,NaN,424399.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,kgnlnvv5


#### First, we want to get the unique count of Participant IDs.

In [7]:
df["participant_id"].nunique()

32

#### Next, we will create a table. We will have a row for each participant and the following columns: 1) Participant ID. 2) First date data was uploaded. 3) Last date data was uploaded. 4) Total number of days GPS data was collected. 5) Average daily GPS data volume. 6) Total number of days Accelerometer data was collected. 7) Average daily Accelerometer data volume.

In [8]:
summary = (
    df.groupby("participant_id", as_index=False) # groups by participant id so that metrics are calculated per participant
    .agg(
        # earliest and latest upload dates for each participant
        first_date_uploaded=("date", "min"),
        last_date_uploaded=("date", "max"),

        # gps metrics
        total_GPS_days=("beiwe_gps_bytes", lambda x: (x > 0).sum()),
        pct_study_days_with_GPS=("beiwe_gps_bytes", lambda x: round((x > 0).mean() * 100, 2)),
        avg_daily_GPS_volume=("beiwe_gps_bytes", lambda x: round(x.loc[x > 0].mean(),3)),
        median_daily_GPS_volume=("beiwe_gps_bytes", lambda x: round(x.loc[x > 0].median(), 3)),

        # accelerometer metrics
        total_accelerometer_days=("beiwe_accelerometer_bytes", lambda x: (x > 0).sum()),
        pct_study_days_with_accelerometer=("beiwe_accelerometer_bytes", lambda x: round((x > 0).mean() * 100, 2)),
        avg_daily_accelerometer_volume=("beiwe_accelerometer_bytes", lambda x: round(x.loc[x > 0].mean(),3)),
        median_daily_accelerometer_volume=("beiwe_accelerometer_bytes", lambda x: round(x.loc[x > 0].median(), 3)),
    )
)
summary.head()

,participant_id,first_date_uploaded,last_date_uploaded,total_GPS_days,pct_study_days_with_GPS,avg_daily_GPS_volume,median_daily_GPS_volume,total_accelerometer_days,pct_study_days_with_accelerometer,avg_daily_accelerometer_volume,median_daily_accelerometer_volume
0,3p4eblir,2026-01-06,2026-01-24,19,100.00,444318.316,431593.0,19,100.0,9.298536e+07,96981111.0
1,443yroib,2025-12-04,2026-01-20,48,100.00,45067.917,43716.5,48,100.0,2.325094e+07,21641896.0
2,555gcdo6,2025-04-08,2025-06-18,71,98.61,59117.113,54847.0,72,100.0,1.744188e+07,16835093.0
3,6a4s9uln,2025-08-11,2025-10-19,70,100.00,349420.329,278617.0,70,100.0,6.483083e+07,59646361.0
4,7evdij49,2025-09-18,2025-11-04,48,100.00,94672.021,86591.5,48,100.0,2.450423e+07,25828348.0


In [10]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

def autosave_outputs_to_pdf(study_id, outputs):
    pdf_file = f"summary_outputs_{study_id}.pdf"
    doc = SimpleDocTemplate(
        pdf_file,
        pagesize=letter,
        rightMargin=0.5*inch,
        leftMargin=0.5*inch,
        topMargin=0.5*inch,
        bottomMargin=0.5*inch,
    )
    styles = getSampleStyleSheet()
    story = []

    for item in outputs:
        # If it's a long multi-line block (like your summary table), render as preformatted text
        if isinstance(item, str) and "\n" in item:
            story.append(Preformatted(item, styles["Code"]))
        else:
            story.append(Paragraph(str(item), styles["BodyText"]))
        story.append(Spacer(1, 6))

    doc.build(story)
    return pdf_file


In [ ]:
from datetime import datetime

# compute the metrics again to build the outputs for the PDF
unique_participants = df["participant_id"].nunique()

# compute the participant-level summary table
summary = (
    df.groupby("participant_id", as_index=False)  # groups by participant id so that metrics are calculated per participant
      .agg(
          # earliest and latest upload dates for each participant
          first_date_uploaded=("date", "min"),
          last_date_uploaded=("date", "max"),

          # gps metrics
          total_GPS_days=("beiwe_gps_bytes", lambda x: (x > 0).sum()),
          pct_study_days_with_GPS=("beiwe_gps_bytes", lambda x: round((x > 0).mean() * 100, 2)),
          avg_daily_GPS_volume=("beiwe_gps_bytes", lambda x: round(x.loc[x > 0].mean(), 3)),
          median_daily_GPS_volume=("beiwe_gps_bytes", lambda x: round(x.loc[x > 0].median(), 3)),

          # accelerometer metrics
          total_accelerometer_days=("beiwe_accelerometer_bytes", lambda x: (x > 0).sum()),
          pct_study_days_with_accelerometer=("beiwe_accelerometer_bytes", lambda x: round((x > 0).mean() * 100, 2)),
          avg_daily_accelerometer_volume=("beiwe_accelerometer_bytes", lambda x: round(x.loc[x > 0].mean(), 3)),
          median_daily_accelerometer_volume=("beiwe_accelerometer_bytes", lambda x: round(x.loc[x > 0].median(), 3)),
      )
)

# choose column subsets so the table fits on the PDF page
gps_cols = [
    "participant_id",
    "first_date_uploaded",
    "last_date_uploaded",
    "total_GPS_days",
    "pct_study_days_with_GPS",
    "avg_daily_GPS_volume",
    "median_daily_GPS_volume",
]

accel_cols = [
    "participant_id",
    "total_accelerometer_days",
    "pct_study_days_with_accelerometer",
    "avg_daily_accelerometer_volume",
    "median_daily_accelerometer_volume",
]

summary_print = summary.rename(columns={
    "participant_id": "pid",
    "first_date_uploaded": "first_date",
    "last_date_uploaded": "last_date",
    "total_GPS_days": "gps_days",
    "pct_study_days_with_GPS": "gps_%days",
    "avg_daily_GPS_volume": "gps_avg",
    "median_daily_GPS_volume": "gps_med",
    "total_accelerometer_days": "acc_days",
    "pct_study_days_with_accelerometer": "acc_%days",
    "avg_daily_accelerometer_volume": "acc_avg",
    "median_daily_accelerometer_volume": "acc_med",
})


# build output strings for the pdf 
gps_cols = ["pid","first_date","last_date","gps_days","gps_%days","gps_avg","gps_med"]
accel_cols = ["pid","acc_days","acc_%days","acc_avg","acc_med"]

outputs = [
    f"Study ID: {study_id}",
    f"PDF created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    f"Unique participants: {unique_participants}",
    "",
    "Participant-level summary table (GPS metrics):",
    summary_print[gps_cols].to_string(index=False),
    "",
    "Participant-level summary table (Accelerometer metrics):",
    summary_print[accel_cols].to_string(index=False),
]


# autosave pdf
pdf_file = autosave_outputs_to_pdf(study_id, outputs)
print(f"Autosaved PDF to: {pdf_file}")





Autosaved PDF to: summary_outputs_XXb2Gm78TVBGesqyd6oPjjXQ.pdf
